# 01 Data Audit

This notebook verifies the official UCI workbook through the reusable `credit_risk.data` module. It does not clean, impute, recode, split, or model the data.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.data import RAW_XLS_NAME, audit_dataset, format_audit_report, load_dataset

In [2]:
data_path = PROJECT_ROOT / 'data' / 'raw' / RAW_XLS_NAME
frame = load_dataset(data_path)
frame.head()

,customer_id,credit_limit,sex,education,marital_status,age,repayment_status_sep,repayment_status_aug,repayment_status_jul,repayment_status_jun,...,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr,default_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
audit = audit_dataset(frame)
print(json.dumps(audit, indent=2))

{
  "shape": {
    "rows": 30000,
    "columns": 25
  },
  "missing_cells": 0,
  "duplicate_rows": 0,
  "duplicate_rows_without_id": 35,
  "unique_customer_ids": 30000,
  "target_counts": {
    "0": 23364,
    "1": 6636
  },
  "target_rate": 0.2212,
  "undocumented_codes": {
    "education": [
      0,
      5,
      6
    ],
    "marital_status": [
      0
    ],
    "repayment_status": {
      "repayment_status_sep": [
        -2,
        0
      ],
      "repayment_status_aug": [
        -2,
        0
      ],
      "repayment_status_jul": [
        -2,
        0
      ],
      "repayment_status_jun": [
        -2,
        0
      ],
      "repayment_status_may": [
        -2,
        0
      ],
      "repayment_status_apr": [
        -2,
        0
      ]
    }
  }
}


In [4]:
print(format_audit_report(frame))

# Data Quality Report

Generated from the verified UCI workbook by `credit_risk.data`.

## Structural checks

- Parsed shape: 30,000 rows and 25 columns.
- Unique customer IDs: 30,000.
- Exact duplicate rows including ID: 0.
- Duplicate rows after dropping ID: 35. These are not automatically removed because distinct IDs can represent different clients with identical observed values.
- No missing cells were detected after parsing the second header row.

## Target

- Non-default (`0`): 23,364.
- Default next month (`1`): 6,636.
- Observed default rate: 22.12%.

The target is moderately imbalanced. Accuracy alone is not an adequate evaluation metric; later stages will include ROC-AUC, PR-AUC, KS, Brier score, and calibration.

## Undocumented source codes

- `education`: [0, 5, 6].
- `marital_status`: [0].
- Repayment status fields:
  - `repayment_status_sep`: [-2, 0]
  - `repayment_status_aug`: [-2, 0]
  - `repayment_status_jul`: [-2, 0]
  - `repayment_status_jun`: [-2, 0]
  - `repayment

## Interpretation boundary

The behavioral variables precede the target, but they describe existing card accounts and are generally unavailable for new-to-bank applicants. Undocumented category codes and negative bill balances are preserved for later analysis rather than silently rewritten.